# 📁 Files & Cryptographic Hashing

The SDK provides dual hashing: **SHA-512** (via Zenroom WASM) for Zenflows,
and **SHA-256** (via Web Crypto API) for DPP attachments.

## Why two hash algorithms?

| | SHA-512 (Zenflows) | SHA-256 (DPP) |
|---|---|---|
| **Engine** | Zenroom (WASM) | Web Crypto API |
| **Use** | File upload proof | Attachment integrity |
| **Browser support** | May fail without COOP/COEP | Always works |
| **Async?** | Yes (WASM init) | Yes (crypto.subtle) |

## Setup

In [ ]:
(async () => {
  (async () => {
    const { InterfacerClient } = await import('https://esm.sh/@dyne/interfacer-client');
    const { SANDBOX_CONFIG } = await import('./config.js');
    
    const client = new InterfacerClient(SANDBOX_CONFIG);
    
    console.log('📁 FileClient ready');
    console.log('🔐 Authenticated:', client.isAuthenticated());    
    // Expose to other cells via window
    if (!window._if) window._if = {};
    window._if.InterfacerClient = InterfacerClient;
    window._if.SANDBOX_CONFIG = SANDBOX_CONFIG;
    window._if.client = client;
  })();
  
  // Expose to other cells via window
  if (!window._if) window._if = {};
  window._if.InterfacerClient = InterfacerClient;
  window._if.SANDBOX_CONFIG = SANDBOX_CONFIG;
  window._if.client = client;
})();


## 1. SHA-512 Hashing (Zenflows)

Zenflows uses SHA-512 hashes computed via Zenroom WASM for file integrity.

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  (async () => {
    // Get shared state
    const _if = window._if || {};
    const InterfacerClient = _if.InterfacerClient;
    const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
    const client = _if.client;
    // Create a test file
    const testContent = JSON.stringify({
      name: 'test-model.stl',
      vertices: 1024,
      description: 'A test 3D model for hashing demo',
    });
    const testFile = new File([testContent], 'test-model.stl', { type: 'application/sla' });
    
    console.log('📄 Test file:', testFile.name, `(${testFile.size} bytes)`);
    
    // Hash for Zenflows (SHA-512 via Zenroom)
    try {
      const hash = await client.files.hashFileForZenflows(testFile);
      console.log('✅ SHA-512 hash:', hash);
    } catch (err) {
      console.warn('⚠️ SHA-512 failed (Zenroom WASM may not be available):', err.message);
    }
  })();
})();


## 2. SHA-256 Hashing (DPP / Web Crypto)

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  (async () => {
    // Get shared state
    const _if = window._if || {};
    const InterfacerClient = _if.InterfacerClient;
    const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
    const client = _if.client;
    try {
      const sha256 = await client.files.hashFileSHA256(testFile);
      console.log('✅ SHA-256 hash:', sha256);
    } catch (err) {
      console.error('❌ SHA-256 failed:', err.message);
    }
  })();
})();


## 3. Prepare Files for Upload

`prepFilesForZenflows` hashes all files and returns descriptors
that can be passed to resource creation mutations.

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  (async () => {
    // Get shared state
    const _if = window._if || {};
    const InterfacerClient = _if.InterfacerClient;
    const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
    const client = _if.client;
    // Create multiple test files
    const files = [
      new File(['model data'], 'enclosure.stl', { type: 'application/sla' }),
      new File(['image data'], 'photo.png', { type: 'image/png' }),
      new File(['doc data'], 'readme.md', { type: 'text/markdown' }),
    ];
    
    try {
      const prepared = await client.files.prepFilesForZenflows(files);
      console.log('✅ Prepared', prepared.length, 'files:');
      
      for (const f of prepared) {
        console.log(`  ${f.name}: hash=${f.hash.substring(0, 12)}..., mime=${f.mimeType}, size=${f.size}`);
      }
      
      console.log('\n📋 ZenflowsFile structure:', Object.keys(prepared[0] || {}).join(', '));
    } catch (err) {
      console.warn('⚠️ Prep failed:', err.message);
    }
  })();
})();


## 4. Upload to Zenflows

Files are uploaded to the Zenflows file storage endpoint.
The endpoint URL is `zenflowsFileUrl` in your config.

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  (async () => {
    // Get shared state
    const _if = window._if || {};
    const InterfacerClient = _if.InterfacerClient;
    const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
    const client = _if.client;
    console.log('📤 Zenflows file URL:', client.config.zenflowsFileUrl || '(not configured)');
    
    const uploadFile = new File(['binary content here'], 'upload-test.bin', { type: 'application/octet-stream' });
    
    try {
      await client.files.uploadToZenflows(uploadFile);
      console.log('✅ Uploaded to Zenflows');
    } catch (err) {
      console.warn('⚠️ Upload failed (sandbox may not accept uploads):', err.message);
    }
    
    // Batch upload
    try {
      await client.files.uploadToZenflowsBatch(files);
      console.log('✅ Batch uploaded', files.length, 'files');
    } catch (err) {
      console.warn('⚠️ Batch upload failed:', err.message);
    }
  })();
})();


## 5. Retrieving Files

Files uploaded to Zenflows are referenced by their hash.
Use `getFileUrl` and `getImageSrc` to generate URLs.

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  (async () => {
    // Get shared state
    const _if = window._if || {};
    const InterfacerClient = _if.InterfacerClient;
    const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
    const client = _if.client;
    // Get proxy URL for a file hash
    const exampleHash = 'abcdef1234567890';
    console.log('🔗 File URL:', client.files.getFileUrl(exampleHash));
    
    // Get image src from GraphQL response
    const gqlImage = {
      hash: exampleHash,
      name: 'photo.png',
      mimeType: 'image/png',
      // If 'bin' is present, returns a data URL
      bin: 'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg==',
    };
    
    console.log('🖼️ Image src:', client.files.getImageSrc(gqlImage).substring(0, 50) + '...');
    
    // Without bin, returns proxy URL
    const gqlImageNoBin = { hash: exampleHash, name: 'photo.png', mimeType: 'image/png' };
    console.log('🔗 Image src (no bin):', client.files.getImageSrc(gqlImageNoBin));
  })();
})();


## 6. Upload to DPP

The DPP (Digital Product Passport) has its own file storage.
Files are signed with the EdDSA key and uploaded with SHA-256 checksums.

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  (async () => {
    // Get shared state
    const _if = window._if || {};
    const InterfacerClient = _if.InterfacerClient;
    const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
    const client = _if.client;
    const dppFile = new File(['DPP attachment content'], 'datasheet.pdf', { type: 'application/pdf' });
    
    try {
      const attachment = await client.files.uploadToDpp(dppFile);
      console.log('✅ DPP attachment uploaded:');
      console.log('  ID:', attachment.id);
      console.log('  Checksum:', attachment.checksum);
      console.log('  Keys:', Object.keys(attachment).join(', '));
    } catch (err) {
      console.warn('⚠️ DPP upload failed (requires auth + dppUrl):', err.message);
    }
  })();
})();


## 7. Upload 3D Model Files to DPP

For 3D printable models (STL, STEP, 3MF, etc.), use `uploadModelsToDpp`
which returns `ProjectModelMetadata` compatible with project creation.

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  (async () => {
    // Get shared state
    const _if = window._if || {};
    const InterfacerClient = _if.InterfacerClient;
    const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
    const client = _if.client;
    const modelFiles = [
      new File(['STL binary data here'], 'enclosure.stl', { type: 'application/sla' }),
      new File(['STEP model data'], 'assembly.step', { type: 'application/step' }),
    ];
    
    try {
      const models = await client.files.uploadModelsToDpp(modelFiles);
      console.log('✅ Models uploaded:', models.length, 'files');
      
      for (const m of models) {
        console.log(`  ${m.fileName}: ${m.extension} (${m.size} bytes) at ${m.url}`);
      }
      
      console.log('\n📋 Model metadata keys:', Object.keys(models[0] || {}).join(', '));
    } catch (err) {
      console.warn('⚠️ Model upload failed:', err.message);
    }
  })();
})();


## 📦 File Type Reference

| Type | Fields |
|---|---|
| **ZenflowsFile** | name, hash, mimeType, extension, size |
| **GqlFile** | hash, name, mimeType, bin? (base64), extension? |
| **DppAttachment** | id, fileName, contentType, url, size, checksum, storage: 'dpp' |
| **ProjectModelMetadata** | id, fileName, contentType, url, size, extension, checksum, uploadedAt, storage |

## ✅ Summary

- ✅ SHA-512 hashing for Zenflows (Zenroom WASM)
- ✅ SHA-256 hashing for DPP (Web Crypto API)
- ✅ File preparation descriptors
- ✅ Zenflows file upload (single + batch)
- ✅ File URL retrieval (`getFileUrl`, `getImageSrc`)
- ✅ DPP attachment upload with EdDSA signing
- ✅ 3D model upload to DPP

**Next:** `04_Digital_Product_Passport_DPP.ipynb` — DPP CRUD operations!